# Day 4: PHI Governance & De-Identification Engine

This notebook demonstrates Format-Preserving Encryption (FPE) using salted hashes, deterministic date-shifting, and Safe Harbor 18 redaction.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sha2, concat, lit, expr, udf
from pyspark.sql.types import StringType

os.environ["EHDIP_HASH_SALT"] = "super_secret_salt"

spark = SparkSession.builder \
    .appName("Day04_DeID") \
    .getOrCreate()


In [ ]:
# Create synthetic raw patient dataframe
data = [
    ("pat_001", "123-45-6789", "MRN999", "1980-05-15", "Patient John Doe presented with fever."),
    ("pat_002", "987-65-4321", "MRN888", "1992-11-20", "Jane Smith has a history of asthma.")
]
columns = ["patient_id", "ssn", "mrn", "birth_date", "clinical_notes"]
df = spark.createDataFrame(data, columns)

print("Raw Data:")
df.show(truncate=False)


In [ ]:
# Apply De-ID
salt = os.getenv("EHDIP_HASH_SALT")

# FPE via Salted Hash
df_deid = df.withColumn("ssn_fpe", sha2(concat(col("ssn"), lit(salt)), 256)) \
            .withColumn("mrn_fpe", sha2(concat(col("mrn"), lit(salt)), 256))

# Date Shifting (-30 to +30 days deterministically based on patient_id)
shift_expr = "cast(abs(hash(patient_id)) % 61 as int) - 30"
df_deid = df_deid.withColumn("birth_date_shifted", expr(f"date_add(cast(birth_date as date), {shift_expr})"))

# Safe Harbor Redaction (Mock)
@udf(returnType=StringType())
def redact_notes(text):
    if not text: return text
    return text.replace("John", "[REDACTED]").replace("Doe", "[REDACTED]").replace("Jane", "[REDACTED]").replace("Smith", "[REDACTED]")

df_deid = df_deid.withColumn("clinical_notes_redacted", redact_notes(col("clinical_notes")))

# Select final de-identified columns
final_deid = df_deid.select("patient_id", "ssn_fpe", "mrn_fpe", "birth_date_shifted", "clinical_notes_redacted")

print("De-identified Data:")
final_deid.show(truncate=False)
